# NLP Pipeline - Practice

## Installer les prérequis

In [1]:
%pip install PyPDF2 nltk scikit-learn pandas requests beautifulsoup4 selenium webdriver-manager spacy
!python -m spacy download fr_core_news_sm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     - 0 bytes ? 0:00:00
     - 0 bytes ? 0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Wheel 'fr-core-news-sm' located at C:\Users\HP\AppData\Local\Temp\pip-unpack-jar3_r3h\fr_core_news_sm-3.8.0-py3-none-any.whl is invalid.


## Exemple 1 : 

In [2]:
import requests
from bs4 import BeautifulSoup
import spacy
import PyPDF2
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Chargement du modèle linguistique français de spaCy
# Ce modèle contient les dictionnaires pour le POS tagging, la lemmatisation et le NER
nlp = spacy.load("fr_core_news_sm")

# -----------------------------------------------------------------------------
# ÉTAPE 1 : COLLECTE DES DONNÉES
# Rassembler le texte brut à partir de sources variées (Web, PDF, etc.)
# -----------------------------------------------------------------------------
def collect_data_web(url):
    """Récupère le texte brut depuis une page HTML."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return " ".join([p.text for p in soup.find_all('p')])

# -----------------------------------------------------------------------------
# ÉTAPE 2 : PRÉTRAITEMENT DU TEXTE
# Prépare le texte : Nettoyage, Tokenisation, Normalisation et Réduction
# -----------------------------------------------------------------------------
def preprocess_text(raw_text):
    # A. NETTOYAGE & NORMALISATION : Passage en minuscule et retrait des espaces inutiles
    text = raw_text.lower().strip()
    
    # B. TOKENISATION & LEMMATISATION : spaCy découpe en tokens et trouve la forme canonique
    doc = nlp(text)
    
    # C. FILTRAGE : On retire les 'Stopwords' (mots vides) et la ponctuation
    # On ne garde que le 'lemme' (racine linguistique) pour réduire la complexité
    tokens_valides = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    
    return doc, " ".join(tokens_valides)

# -----------------------------------------------------------------------------
# ÉTAPE 3 & 4 : ANALYSE SYNTAXIQUE & SÉMANTIQUE
# Comprendre la structure (POS: Part of Speech) et identifier les entités (NER)
#          --> Part-of-Speech (POS) tagging is the process of assigning grammatical categories 
#             (like noun, verb, adjective, etc.) to each word in a sentence based on its definition and context. 
# -----------------------------------------------------------------------------
def analyze_linguistics(doc):
    print(f"{'Mot':<12} | {'Nature (POS)':<10} | {'Dépendance'}")
    print("-" * 40)
    for token in list(doc)[:5]: # Exemple sur les 5 premiers mots
        # ÉTAPE 3 : POS Tagging (Nom, Verbe, Adjectif...)
        print(f"{token.text:<12} | {token.pos_:<10} | {token.dep_}")

    # ÉTAPE 4 : Reconnaissance d'Entités Nommées (NER)
    print("\n--- Entités Nommées détectées ---")
    for ent in doc.ents:
        print(f"Entité : {ent.text} ({ent.label_})")

# -----------------------------------------------------------------------------
# ÉTAPE 5 : REPRÉSENTATION DES DONNÉES (VECTORISATION)
# Convertir le texte en nombres pour que la machine puisse "calculer"
# -----------------------------------------------------------------------------
def vectorize_data(corpus):
    # Utilisation de TF-IDF pour donner du poids aux mots significatifs
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(corpus)
    return X, vectorizer

# -----------------------------------------------------------------------------
# ÉTAPE 6 & 7 : MODÉLISATION ET ÉVALUATION
# Appliquer un algorithme et tester ses performances
# -----------------------------------------------------------------------------
'''
def train_and_evaluate(X, y):
    # Exemple de classification (Spam vs Non-Spam)
    model = MultinomialNB()
    model.fit(X, y)
    
    # Évaluation (Précision, Rappel, F1-score...)
    score = model.score(X, y)
    print(f"\nPrécision du modèle : {score * 100:.2f}%")
    return model
'''
# =============================================================================
# EXÉCUTION DU PIPELINE
# =============================================================================

# Texte d'exemple (Collecte)
raw_content = "Apple a été fondée par Steve Jobs en Californie. Il a créé l'iPhone."

# Prétraitement (Étape 2)
document_spacy, text_clean = preprocess_text(raw_content)

# Analyse (Étape 3 & 4)
analyze_linguistics(document_spacy)

# Représentation (Étape 5)
# (Ici sur un mini-corpus pour l'exemple)
X_vectors, feat_vec = vectorize_data([text_clean])


print("\nPipeline terminé avec succès.")

Mot          | Nature (POS) | Dépendance
----------------------------------------
apple        | NOUN       | nsubj:pass
a            | AUX        | aux:tense
été          | AUX        | aux:pass
fondée       | VERB       | ROOT
par          | ADP        | case

--- Entités Nommées détectées ---
Entité : apple (ORG)
Entité : steve jobs (PER)
Entité : californie (LOC)

Pipeline terminé avec succès.



1. Tokenisation (Le découpage)
Le texte est segmenté en unités atomiques. Le modèle fr_core_news_sm est assez intelligent pour comprendre que "d'ia" doit être séparé en ["d'", "ia"].

2. Stopwords (Le nettoyage)
Le code utilise la propriété .is_stop de spaCy. Les mots comme "les", "de", "la" ou "à" sont évacués car ils sont statistiquement trop fréquents pour aider à classer le texte.

3. Lemmatisation (L'unification)
C'est ici que l'analyse linguistique prend tout son sens. Le mot "étudient" est ramené à "étudier" et "ingénieurs" devient "ingénieur".

Pourquoi ? Si vous avez un autre texte qui parle d'un "ingénieur", le modèle saura qu'il s'agit du même sujet, même si le pluriel diffère.

4. TF-IDF (La signature numérique)
Le tableau final montre le poids de chaque mot.

TF (Term Frequency) : Plus le mot "IA" est présent dans votre texte, plus son score monte.

IDF (Inverse Document Frequency) : Si le mot "FST" est présent dans tous les documents de votre présentation, son score baissera car il n'est plus discriminant (il ne permet plus de différencier un document d'un autre).

Résultat final : Votre phrase est passée d'une structure complexe et humaine à un vecteur de probabilités mathématiques prêt à être injecté dans un classifieur (comme une Régression Logistique ou Naive Bayes).

# Exemple 2: 

In [3]:
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Chargement du modèle français
nlp = spacy.load("fr_core_news_sm")

def trace_nlp_pipeline(text):
    print(f"--- TEXTE BRUT ---\n'{text}'\n")
    
    # 1. NORMALISATION & TOKENISATION
    doc = nlp(text.lower().strip())
    tokens_bruts = [token.text for token in doc]
    print(f"1. Tokenisation (Minuscules) :\n{tokens_bruts}\n")
    
    # 2. FILTRAGE (Stopwords & Ponctuation)
    tokens_filtres = [token.text for token in doc if not token.is_stop and not token.is_punct]
    stopwords_elimines = [token.text for token in doc if token.is_stop]
    print(f"2. Après retrait des Stopwords :\n{tokens_filtres}")
    print(f"(Mots supprimés : {list(set(stopwords_elimines))})\n")
    
    # 3. LEMMATISATION
    # On transforme chaque mot filtré en sa forme dictionnaire (lemme)
    lemmes = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    print(f"3. Après Lemmatisation (Racines) :\n{lemmes}\n")
    
    return " ".join(lemmes)

# --- EXEMPLE DÉTAILLÉ ---
phrase_test = "Les ingénieurs étudient les algorithmes d'IA à la FST de Tanger !"
texte_propre = trace_nlp_pipeline(phrase_test)

# --- 4. ZOOM SUR TF-IDF (Vectorisation) ---
# Imaginons un mini-corpus pour calculer les poids
corpus = [texte_propre, "étudiant fst tanger", "algorithme ia complexe"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Affichage des scores pour le premier document
print("4. Représentation Numérique (TF-IDF) pour le document 1 :")
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df_tfidf.iloc[0].sort_values(ascending=False))

--- TEXTE BRUT ---
'Les ingénieurs étudient les algorithmes d'IA à la FST de Tanger !'

1. Tokenisation (Minuscules) :
['les', 'ingénieurs', 'étudient', 'les', 'algorithmes', "d'", 'ia', 'à', 'la', 'fst', 'de', 'tanger', '!']

2. Après retrait des Stopwords :
['ingénieurs', 'étudient', 'algorithmes', 'ia', 'fst', 'tanger']
(Mots supprimés : ['les', 'la', 'de', "d'", 'à'])

3. Après Lemmatisation (Racines) :
['ingénieur', 'étudier', 'algorithme', 'ia', 'fst', 'tanger']

4. Représentation Numérique (TF-IDF) pour le document 1 :
étudier       0.481482
ingénieur     0.481482
algorithme    0.366180
fst           0.366180
tanger        0.366180
ia            0.366180
complexe      0.000000
étudiant      0.000000
Name: 0, dtype: float64


## Interprétation du résultat

Ce résultat montre la transformation complète d'une phrase humaine en une **signature mathématique** que l'IA peut traiter. Voici l'explication détaillée de chaque étape de votre pipeline :



### 1. Tokenisation (Le découpage)
Le texte brut est segmenté en unités minimales appelées **tokens**. 
* **Observation :** Le modèle a intelligemment séparé `"d'ia"` en `"d'"` et `"ia"`. 
* **Objectif :** Isoler chaque élément syntaxique pour pouvoir les traiter individuellement. La mise en minuscules évite que l'ordinateur ne traite "Les" et "les" comme deux mots différents.


### 2. Retrait des Stopwords (Le filtrage)
On élimine les "mots vides" (déterminants, prépositions). 
* **Observation :** Les mots `['de', 'les', 'la', 'à', "d'"]` ont disparu. 
* **Objectif :** Réduire le "bruit". Ces mots sont statistiquement trop fréquents dans la langue française pour aider à distinguer le sujet de la phrase. On ne garde que les **mots porteurs de sens** (les mots pleins).


### 3. Lemmatisation (L'unification)
C'est l'étape de normalisation linguistique. On ramène chaque mot à sa forme canonique (celle du dictionnaire).
* **Transformations :** * `ingénieurs` (pluriel) $\rightarrow$ **ingénieur** (singulier).
    * `étudient` (verbe conjugué) $\rightarrow$ **étudier** (infinitif).
* **Objectif :** Regrouper les variantes d'un même mot. Si un autre texte utilisait "ingénieur" au singulier, le modèle comprendrait qu'il s'agit du même concept.


### 4. TF-IDF (La pondération mathématique)
C'est ici que l'on transforme les mots en nombres. Le score **TF-IDF** indique l'importance d'un mot dans votre document par rapport à un ensemble de documents (le corpus).

#### Pourquoi certains scores sont-ils plus hauts (0.48 vs 0.36) ?
Le score dépend de deux facteurs :
1.  **TF (Term Frequency) :** Combien de fois le mot apparaît dans votre phrase.
2.  **IDF (Inverse Document Frequency) :** À quel point le mot est **rare** dans les autres documents de votre exemple.

* **Poids fort (0.48) pour "étudier" et "ingénieur" :** Cela signifie que ces mots sont très spécifiques à votre document 1 et n'apparaissent probablement pas (ou très peu) dans les autres documents de comparaison que vous avez fournis au `TfidfVectorizer`. Ils sont donc jugés **très représentatifs** du contenu.
* **Poids moyen (0.36) pour "ia", "fst", "tanger" :** Ces mots sont importants, mais s'ils apparaissent aussi dans les autres documents du corpus (comme "étudiant fst tanger"), leur score baisse car ils deviennent moins "uniques" pour différencier le document 1 des autres.
* **Score 0.00 pour "complexe" et "étudiant" :** Ces mots n'existent tout simplement pas dans votre phrase de test, même s'ils font partie du vocabulaire global du corpus.



### En bref
Votre phrase n'est plus du texte pour la machine, c'est un **vecteur dans un espace multidimensionnel**. Plus le score est élevé, plus le mot est considéré comme un "mot-clé" crucial pour caractériser votre document.

# Exemple 3

In [4]:
import spacy
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Chargement des outils
nlp = spacy.load("fr_core_news_sm")
stemmer = SnowballStemmer("french")

text_brut = "Les algorithmes d'IA analysent les données massives pour les chercheurs à Tanger."

# --- STEP 1 : Tokenisation & Stopwords ---
doc = nlp(text_brut.lower())
tokens_clean = [t for t in doc if not t.is_stop and not t.is_punct]
print(f"1. Après Stopwords : {[t.text for t in tokens_clean]}")

# --- STEP 2 : Stemming ---
stems = [stemmer.stem(t.text) for t in tokens_clean]
print(f"2. Stemming (Racines) : {stems}")

# --- STEP 3 : Lemmatisation ---
lemmas = [t.lemma_ for t in tokens_clean]
print(f"3. Lemmatisation (Dictionnaire) : {lemmas}")

# --- STEP 4 : TF-IDF ---
# On crée un mini-corpus pour que le calcul TF-IDF ait du sens
corpus = [" ".join(lemmas), "ia tanger fst", "algorithme donnée analyse"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Affichage des poids pour notre phrase (Document 0)
print("\n4. Poids TF-IDF (Importance des mots) :")
df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df.iloc[0].sort_values(ascending=False))

1. Après Stopwords : ['algorithmes', 'ia', 'analysent', 'données', 'massives', 'chercheurs', 'tanger']
2. Stemming (Racines) : ['algorithm', 'ia', 'analysent', 'don', 'massiv', 'chercheur', 'tang']
3. Lemmatisation (Dictionnaire) : ['algorithme', 'ia', 'analyser', 'donnée', 'massif', 'chercheur', 'tanger']

4. Poids TF-IDF (Importance des mots) :
analyser      0.433816
massif        0.433816
chercheur     0.433816
algorithme    0.329928
donnée        0.329928
tanger        0.329928
ia            0.329928
analyse       0.000000
fst           0.000000
Name: 0, dtype: float64


# Devoir 

- Extraire les données depuis un pdf
- Utiliser des textes arabes au lieu de francais

In [5]:
import PyPDF2
import nltk
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import re

# Téléchargement des ressources NLTK nécessaires
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

def extract_text_from_pdf(pdf_path="test.pdf"):
    """
    Extrait le texte d'un fichier PDF situé dans le même répertoire.
    """
    text = ""
    try:
        # Ouverture du fichier en mode lecture binaire
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + " "
        return text
    except FileNotFoundError:
        return "Erreur : Le fichier 'test.pdf' est introuvable dans le dossier actuel."

# --- ÉTAPE 1 : COLLECTE DES DONNÉES ---
raw_arabic_text = extract_text_from_pdf("test.pdf")

# Texte de secours si le PDF est vide ou introuvable
if "Erreur" in raw_arabic_text or not raw_arabic_text.strip():
    raw_arabic_text = "المهندسون يدرسون خوارزميات الذكاء الاصطناعي في كلية العلوم والتقنيات بطنجة."

print(f"--- TEXTE BRUT ---\n'{raw_arabic_text}'\n")

# --- ÉTAPE 2 : PRÉTRAITEMENT (NLP ARABE) ---
stemmer = ISRIStemmer()
stop_words_ar = set(stopwords.words('arabic'))

# Nettoyage : suppression des caractères non arabes
text_clean = re.sub(r'[^\w\s\u0600-\u06FF]', '', raw_arabic_text)

# Tokenisation
tokens = nltk.word_tokenize(text_clean)

# Suppression des Stopwords
tokens_sans_stop = [word for word in tokens if word not in stop_words_ar]
print(f"1. Après Stopwords : {tokens_sans_stop}\n")

# Stemming
racines = [stemmer.stem(word) for word in tokens_sans_stop]
print(f"2. Stemming (Racines arabes) : {racines}\n")

# --- ÉTAPE 3 : REPRÉSENTATION (TF-IDF) ---
document_traite = " ".join(racines)

corpus = [
    document_traite,
    "طنجة هندس ذكاء بيانات",
    "تحليل خوارزم حوسب"
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

print("3. Poids TF-IDF (Importance des mots) :")
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print(df_tfidf.iloc[0].sort_values(ascending=False))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


--- TEXTE BRUT ---
'يعتبر الذكاء االصطناعي من أهم التقنيات الحديثة التي تدرس في كلية العلوم والتقنيات بطنجة. يقوم الطالب والباحثون بتحليل البيانات  
الضخمة وتطوير خوارزميات متقدمة تساهم في حل المشكالت المعقدة وتوقع المخاطر المستقبلية في المنطقة   '

1. Après Stopwords : ['يعتبر', 'الذكاء', 'االصطناعي', 'أهم', 'التقنيات', 'الحديثة', 'تدرس', 'كلية', 'العلوم', 'والتقنيات', 'بطنجة', 'يقوم', 'الطالب', 'والباحثون', 'بتحليل', 'البيانات', 'الضخمة', 'وتطوير', 'خوارزميات', 'متقدمة', 'تساهم', 'حل', 'المشكالت', 'المعقدة', 'وتوقع', 'المخاطر', 'المستقبلية', 'المنطقة']

2. Stemming (Racines arabes) : ['عبر', 'ذكء', 'االصطناعي', 'اهم', 'تقن', 'حدث', 'درس', 'كلة', 'علم', 'تقن', 'طنج', 'يقم', 'طلب', 'بحث', 'حلل', 'بين', 'ضخم', 'طور', 'خوارزم', 'تقدم', 'تسا', 'حل', 'شكل', 'عقد', 'وقع', 'خطر', 'مستقبلية', 'نطق']

3. Poids TF-IDF (Importance des mots) :
تقن          0.367742
االصطناعي    0.183871
اهم          0.183871
بين          0.183871
بحث          0.183871
تقدم         0.183871
تسا          0.183871
ح